# Notebook 3: Explainability Evaluation
**Answers RQ2:** How do SHAP and LIME differ in explanation stability and consistency?

## Steps
1. Load trained XGBoost model and preprocessor
2. Fit SHAP and LIME explainers
3. Visualise explanations for sample transactions
4. Run perturbation-based stability analysis
5. Compute SHAP–LIME agreement (Spearman correlation)

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from utils.data_loader import DataLoader, LABEL_COLUMN
from utils.preprocessing import Preprocessor
from explainability.shap_explainer import SHAPExplainer
from explainability.lime_explainer import LIMEExplainer
from explainability.stability_analyzer import StabilityAnalyzer

os.makedirs('../outputs', exist_ok=True)
%matplotlib inline

## 1. Load Models and Test Data

In [ ]:
SAVED_DIR = '../models/saved'
xgb_model  = joblib.load(os.path.join(SAVED_DIR, 'xgb_model.pkl'))['model']
preprocessor = Preprocessor.load(os.path.join(SAVED_DIR, 'preprocessor.pkl'))

loader = DataLoader()
df = loader.load()
_, _, test_df = loader.split(df)

X_test, y_test = preprocessor.transform(test_df)
print(f'Test set: {X_test.shape}')

## 2. Fit SHAP Explainer

In [ ]:
# Use 200 background samples for SHAP
background = X_test[:200]
shap_exp = SHAPExplainer(model=xgb_model)
shap_exp.fit(background)
print('SHAP explainer ready')

## 3. SHAP Waterfall for a Fraud Transaction

In [ ]:
fraud_idx = np.where(y_test == 1)[0][0]
X_fraud = X_test[fraud_idx:fraud_idx+1]

shap_values = shap_exp.explainer(X_fraud)
shap.plots.waterfall(shap_values[0], show=False)
plt.title('SHAP Waterfall – Fraud Transaction')
plt.tight_layout()
plt.savefig('../outputs/shap_waterfall_fraud.png', dpi=150, bbox_inches='tight')
plt.show()

result = shap_exp.explain(X_fraud)
print('Top SHAP features:', result['top_features'])

## 4. SHAP Summary Plot (Global)

In [ ]:
shap_all = shap_exp.explainer(X_test[:500])
shap.summary_plot(shap_all.values, X_test[:500],
                  feature_names=['amount','user_age','merchant_category','device_type'],
                  show=False)
plt.savefig('../outputs/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Fit LIME and Explain Same Transaction

In [ ]:
lime_exp = LIMEExplainer(model=xgb_model)
lime_exp.fit(X_test[:500])

lime_result = lime_exp.explain(X_fraud)
print('LIME top features:', lime_result['top_features'])

# Bar plot of LIME weights
weights = lime_result['local_weights']
fig, ax = plt.subplots(figsize=(6, 3))
labels = list(weights.keys())
vals   = list(weights.values())
colors = ['firebrick' if v > 0 else 'steelblue' for v in vals]
ax.barh(labels, vals, color=colors)
ax.set_title('LIME Local Explanation – Fraud Transaction')
ax.set_xlabel('Feature Weight')
plt.tight_layout()
plt.savefig('../outputs/lime_explanation_fraud.png', dpi=150)
plt.show()

## 6. Stability Analysis (RQ2)

In [ ]:
analyzer = StabilityAnalyzer(
    shap_explainer=shap_exp,
    lime_explainer=lime_exp,
    n_perturbations=100,
)

# Analyse a mix of fraud and legitimate transactions
legit_idx = np.where(y_test == 0)[0][0]

print('\n--- Fraud Transaction ---')
fraud_report = analyzer.analyze(X_fraud, noise_std=0.05)
for k, v in fraud_report.items():
    print(f'  {k}: {v}')

print('\n--- Legitimate Transaction ---')
legit_report = analyzer.analyze(X_test[legit_idx:legit_idx+1], noise_std=0.05)
for k, v in legit_report.items():
    print(f'  {k}: {v}')

## 7. SHAP vs LIME Feature Importance Comparison

In [ ]:
from scipy.stats import spearmanr

feature_names = ['amount', 'user_age', 'merchant_category', 'device_type']

# Compute mean absolute importance over 100 test samples
shap_imp = np.mean(np.abs(shap_all.values[:100]), axis=0)

lime_imps = []
for i in range(min(100, len(X_test))):
    res = lime_exp.explain(X_test[i:i+1])
    lime_imps.append(list(res.get('local_weights', {0:0,1:0,2:0,3:0}).values()))
lime_imp = np.mean(np.abs(lime_imps), axis=0)

corr, pval = spearmanr(shap_imp, lime_imp[:len(shap_imp)])
print(f'SHAP–LIME Spearman correlation: {corr:.4f} (p={pval:.4f})')

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(feature_names))
w = 0.35
ax.bar(x - w/2, shap_imp, width=w, label='SHAP', color='steelblue')
ax.bar(x + w/2, lime_imp[:len(feature_names)], width=w, label='LIME', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(feature_names)
ax.set_ylabel('Mean |Importance|')
ax.set_title(f'SHAP vs LIME Feature Importance (Spearman r={corr:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/shap_vs_lime_importance.png', dpi=150)
plt.show()